In [8]:
import os
import chromadb
from chromadb.errors import InvalidCollectionException
from tqdm import tqdm
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
client = chromadb.Client()

In [3]:
DATA_DIR = "BoPhapDienDienTu/vbpl"

embedding_model = HuggingFaceEmbeddings(model_name="bkai-foundation-models/vietnamese-bi-encoder")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, # chunk size 256 due embedding model limiations
    chunk_overlap=20 
)

g:\anaconda\anaconda3\envs\chatbot-agent\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [11]:
# Helper for data ingestion

def get_data(data_dir):
    """Chunk the data and add metadata"""
    documents = []
    files = os.listdir(data_dir)

    for file in tqdm(files):
        path = os.path.join(data_dir, file)
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Chunk the text
        chunks = text_splitter.split_text(content)

        for chunk in chunks:
            document = Document(
                page_content=chunk,
                metadata={"file_path": path}
            )
            
            documents.append(document)
    
    return documents


def get_vectorstore(client, embedding_model, collection_name, text_dir, persist_dir=None):
    """Load the vectorstore. If not created, embed and add the documents to the collection

    Args:
        client: Chroma ClientAPI
        embedding_model: Any embedding model
        collection_name: name of the collection
        text_dir: directory of html files
        persist_dir: persistant folder

    """
    try:
        client.get_collection(collection_name)

        # If collection exist (already ingested), simply load the chroma
        vectorstore = Chroma(
            client=client,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir
        )

    except InvalidCollectionException: # collection does not exist
        # Create new collection and ingest the data
        doc_list = get_data(text_dir)
        import time
        print("Start inset:", time.time())
        vectorstore = Chroma.from_documents(
            client=client,
            documents=doc_list,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir
        )

    return vectorstore

In [12]:
vector_store = get_vectorstore(client=client, embedding_model=embedding_model, collection_name="isods", text_dir=DATA_DIR)

100%|██████████| 5943/5943 [42:43<00:00,  2.32it/s]  


Start inset: 1739169629.6263137


KeyboardInterrupt: 

In [13]:
# Semantic search vector store
k = 1
results = vector_store.similarity_search_by_vector(query="Quy định về sử dụng mũ bảo hiểm khi lái xe",k=k)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

NameError: name 'vector_store' is not defined